# Emerging Artist Discovery: ListenBrainz Cleaning Validation

This notebook documents and reproduces the approved raw-data validation and cleaning workflow for the **Emerging Artist Discovery Through Listening Behavior** project. The project explores whether listening patterns can help identify artists for additional exposure and listener groups likely to continue engaging with them.

This is a cleaning and validation workflow, not an artist-ranking exercise. It does not classify an artist as emerging or assign a final continued-engagement label. The cells below read the raw data, reproduce the approved in-memory logic, and compare that result with `cleaned_listenbrainz.csv`; they do not write or overwrite any files.


## What a row represents and raw-data baseline

One row represents **one user's listen to one track at one point in time**. A person can listen to the same recording more than once, so repeated recording identifiers can be legitimate separate events.

The raw ListenBrainz sample has **2,214 rows and 9 columns**, covering five users:

| User | Raw listen rows |
|---|---:|
| `iliekcomputers` | 500 |
| `H-music` | 500 |
| `s_john_s` | 500 |
| `John V` | 500 |
| `music_af` | 214 |

The source columns are `username`, `listened_at`, `track_name`, `artist_name`, `release_name`, `recording_msid`, `artist_mbids`, `recording_mbid`, and `release_mbid`. CSV itself has no declared types: source values are loaded as text; `listened_at` contains integer-form Unix seconds, and populated `artist_mbids` cells contain JSON arrays. The inspected missing-value baseline is: `release_name` 1,027 blank (46.39%); `artist_mbids` and `release_mbid` 974 each (43.99%); `recording_mbid` 943 (42.59%); and zero blank values in the other five source columns.


## Approved decisions and why they matter

- **Missing optional metadata is retained.** Missing `release_name`, `artist_mbids`, `recording_mbid`, or `release_mbid` does not make a listen event invalid. Do not fill values or infer identifiers; dropping unresolved listens would also skew user coverage.
- **Repeated identifiers are not duplicate listens.** A recording or release ID identifies a music entity, not a unique listen event. Keep repeats; only an exact repeat across all nine raw event fields is eligible for duplicate removal. The inspected raw file has no exact duplicate rows.
- **Artist names remain exactly as submitted.** Preserve case, spelling, punctuation, and spacing. Do not automatically normalize, merge similar names, or infer MBIDs. Use an available MBID as the preferred resolved identity and keep unresolved relationships flagged.
- **The primary event table stays at one row per listen.** Parse `artist_mbids` into a separate derived array representation. Do not explode the event table; use a separate event-to-artist bridge if needed later.
- **Raw timestamps and text remain intact.** Add a separate UTC datetime from `listened_at`; retain the raw epoch string. Retain the original `artist_mbids` JSON text alongside its parsed representation.
- **Time coverage limits growth conclusions.** User windows do not overlap, and four users reached the 500-listen cap. Record each user's observed window and cap status; do not force a shared window during cleaning or claim synchronized cross-user growth from this sample.
- **No MusicBrainz merge occurs here.** The metadata dataset is not present. The intended future relationship is listen → resolved artist MBID → MusicBrainz artist metadata.
- **Thresholds are deferred.** Do not classify emerging artists or assign a final continued-engagement label during cleaning. Repeated listens and listens on distinct days may be described as evidence during later analysis.


In [1]:
from pathlib import Path
import json
import re

import pandas as pd

RAW_PATH = Path("listenbrainz_sample.csv")
CLEANED_PATH = Path("cleaned_listenbrainz.csv")
RAW_COLUMNS = [
    "username",
    "listened_at",
    "track_name",
    "artist_name",
    "release_name",
    "recording_msid",
    "artist_mbids",
    "recording_mbid",
    "release_mbid",
]
EVENT_KEY_COLUMNS = ["username", "listened_at", "recording_msid"]
SAMPLE_CAP = 500
EXPECTED_ROWS = 2_214
UUID_PATTERN = re.compile(
    r"^[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}$"
)


## Load raw values without type or missing-token rewriting

`dtype=str` and `keep_default_na=False` keep names and IDs as literal text. Empty CSV cells remain empty strings for this comparison; missing-value counts below explicitly treat only an empty source cell as missing. The raw file is never opened for writing.


In [2]:
raw = pd.read_csv(
    RAW_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
)

assert list(raw.columns) == RAW_COLUMNS, "Unexpected raw CSV schema"
print("Raw shape:", raw.shape)
print("Raw columns:", raw.columns.tolist())
print("Loaded dtypes:")
print(raw.dtypes)


Raw shape: (2214, 9)
Raw columns: ['username', 'listened_at', 'track_name', 'artist_name', 'release_name', 'recording_msid', 'artist_mbids', 'recording_mbid', 'release_mbid']
Loaded dtypes:
username          object
listened_at       object
track_name        object
artist_name       object
release_name      object
recording_msid    object
artist_mbids      object
recording_mbid    object
release_mbid      object
dtype: object


In [3]:
print("Raw rows:", len(raw))
print("Unique users:", raw["username"].nunique())
print("Listen counts by user:")
print(raw["username"].value_counts().sort_index())
print("Distinct submitted artist_name values:", raw["artist_name"].nunique())


Raw rows: 2214
Unique users: 5
Listen counts by user:
username
H-music           500
John V            500
iliekcomputers    500
music_af          214
s_john_s          500
Name: count, dtype: int64
Distinct submitted artist_name values: 795


## Missing-value baseline

Optional metadata is intentionally left missing. This check reports the empty-cell counts for every raw column; it does not fill, infer, or remove any values.


In [4]:
source_missing = raw.eq("").sum()
source_missing_percent = (source_missing / len(raw) * 100).round(2)
missing_baseline = pd.DataFrame({
    "missing_count": source_missing,
    "missing_percent": source_missing_percent,
})
missing_baseline


,missing_count,missing_percent
username,0,0.00
listened_at,0,0.00
track_name,0,0.00
artist_name,0,0.00
release_name,1027,46.39
recording_msid,0,0.00
artist_mbids,974,43.99
recording_mbid,943,42.59
release_mbid,974,43.99


## Exact duplicates and candidate event key

An exact duplicate is defined as a repeated row across all nine raw source fields. The candidate event key is `username` + `listened_at` + `recording_msid`; its uniqueness is checked separately. Repeated recording, release, or artist IDs alone are not grounds for removal. The next cell explicitly constructs an in-memory event table by excluding only exact duplicate rows, so any row-count difference is visible and reconciled.


In [5]:
exact_duplicate_mask = raw.duplicated(subset=RAW_COLUMNS, keep="first")
exact_duplicate_rows = int(exact_duplicate_mask.sum())

source_event_key_duplicates = int(
    raw.duplicated(subset=EVENT_KEY_COLUMNS, keep=False).sum()
)

# This is the approved exact-duplicate rule, applied only in memory.
cleaned = raw.loc[~exact_duplicate_mask, RAW_COLUMNS].copy()

print("Source rows:", len(raw))
print("Exact duplicate rows:", exact_duplicate_rows)
print("Rows removed under exact-duplicate rule:", len(raw) - len(cleaned))
print("Candidate event-key duplicate rows:", source_event_key_duplicates)
print("Candidate event key unique:", source_event_key_duplicates == 0)


Source rows: 2214
Exact duplicate rows: 0
Rows removed under exact-duplicate rule: 0
Candidate event-key duplicate rows: 0
Candidate event key unique: True


## Add a derived UTC timestamp

Convert the raw Unix-seconds value into a separate UTC-aware datetime column. Keep `listened_at` unchanged. Failed conversions would remain as missing values in the derived column and the corresponding events would still be retained.


In [6]:
cleaned["listened_at_utc"] = pd.to_datetime(
    pd.to_numeric(cleaned["listened_at"], errors="coerce"),
    unit="s",
    utc=True,
    errors="coerce",
)

timestamp_failures = int(cleaned["listened_at_utc"].isna().sum())
valid_timestamps = cleaned["listened_at_utc"].dropna()
print("Timestamp conversion failures:", timestamp_failures)
print("Earliest UTC timestamp:", valid_timestamps.min())
print("Latest UTC timestamp:", valid_timestamps.max())


Timestamp conversion failures: 0
Earliest UTC timestamp: 2019-03-07 13:17:39+00:00
Latest UTC timestamp: 2026-10-02 22:25:37+00:00


## Parse artist MBIDs safely without expanding events

The original `artist_mbids` cell stays untouched. For validation, parse populated JSON values into lists and check the array and UUID-shaped ID elements. Add `artist_mbids_parsed` as a separate JSON-serialized array representation in the in-memory event table. A blank source value remains blank in the derived field; no identity is inferred.


In [7]:
def parse_artist_mbids(value):
    if value == "":
        return None, "missing", 0
    try:
        parsed = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return None, "invalid_json", 0
    if not isinstance(parsed, list):
        return None, "not_array", 0
    invalid_ids = sum(
        not isinstance(mbid, str) or not UUID_PATTERN.fullmatch(mbid)
        for mbid in parsed
    )
    return parsed, "valid_array", int(invalid_ids)

mbid_results = [parse_artist_mbids(value) for value in raw["artist_mbids"]]
mbid_validation = {
    "valid_arrays": sum(status == "valid_array" for _, status, _ in mbid_results),
    "invalid_json": sum(status == "invalid_json" for _, status, _ in mbid_results),
    "not_arrays": sum(status == "not_array" for _, status, _ in mbid_results),
    "invalid_ids": sum(invalid_ids for _, _, invalid_ids in mbid_results),
}
parsed_json = [
    json.dumps(parsed, ensure_ascii=False) if parsed is not None else ""
    for parsed, _, _ in mbid_results
]
cleaned["artist_mbids_parsed"] = [parsed_json[index] for index in cleaned.index]

all_artist_mbids = [
    mbid
    for parsed, status, _ in mbid_results
    if status == "valid_array"
    for mbid in parsed
    if isinstance(mbid, str)
]
artist_mbid_summary = {
    "rows_with_populated_artist_mbids": int(raw["artist_mbids"].ne("").sum()),
    "rows_missing_artist_mbids": int(raw["artist_mbids"].eq("").sum()),
    "ID_occurrences": len(all_artist_mbids),
    "distinct_artist_MBIDs": len(set(all_artist_mbids)),
    **mbid_validation,
}
artist_mbid_summary


{'rows_with_populated_artist_mbids': 1240,
 'rows_missing_artist_mbids': 974,
 'ID_occurrences': 1538,
 'distinct_artist_MBIDs': 611,
 'valid_arrays': 1240,
 'invalid_json': 0,
 'not_arrays': 0,
 'invalid_ids': 0}

## Repeated entity IDs, separate from duplicate events

Repeated IDs can indicate a recording played many times or multiple listens to a release. This diagnostic reports identifier repetition separately and does not remove any event.


In [8]:
def repeated_id_summary(frame, column):
    values = frame.loc[frame[column].ne(""), column]
    frequencies = values.value_counts()
    repeated = frequencies[frequencies > 1]
    return {
        "distinct_populated_ids": int(frequencies.size),
        "IDs_repeated": int(len(repeated)),
        "rows_with_repeated_ID": int(repeated.sum()),
        "occurrences_beyond_first": int((repeated - 1).sum()),
    }

pd.DataFrame({
    column: repeated_id_summary(cleaned, column)
    for column in ["recording_msid", "recording_mbid", "release_mbid"]
}).T


,distinct_populated_ids,IDs_repeated,rows_with_repeated_ID,occurrences_beyond_first
recording_msid,1525,345,1034,689
recording_mbid,930,188,529,341
release_mbid,631,195,804,609


## Per-user observation-window summary

The row count is based on retained event rows. The 500-listen-cap flag uses each user's raw source row count, so exact-duplicate handling cannot conceal that the sample cap was reached. Durations are based on valid derived UTC timestamps.


In [9]:
source_counts_by_user = raw.groupby("username", dropna=False).size().to_dict()
summary_rows = []
for username in sorted(raw["username"].unique()):
    user_events = cleaned.loc[cleaned["username"].eq(username)]
    user_timestamps = user_events["listened_at_utc"].dropna()
    first_listen = user_timestamps.min() if not user_timestamps.empty else pd.NaT
    last_listen = user_timestamps.max() if not user_timestamps.empty else pd.NaT
    duration = (
        last_listen - first_listen
        if pd.notna(first_listen) and pd.notna(last_listen)
        else pd.NaT
    )
    source_row_count = int(source_counts_by_user.get(username, 0))
    summary_rows.append({
        "username": username,
        "row_count": int(len(user_events)),
        "first_listen_utc": first_listen,
        "last_listen_utc": last_listen,
        "observed_duration": duration,
        "source_row_count": source_row_count,
        "sampling_cap_reached": source_row_count >= SAMPLE_CAP,
    })

user_observation_summary = pd.DataFrame(summary_rows)
user_observation_summary


,username,row_count,first_listen_utc,last_listen_utc,observed_duration,source_row_count,sampling_cap_reached
0,H-music,500,2019-07-15 16:56:10+00:00,2019-11-15 18:03:47+00:00,123 days 01:07:37,500,True
1,John V,500,2026-09-25 03:38:33+00:00,2026-10-02 22:25:37+00:00,7 days 18:47:04,500,True
2,iliekcomputers,500,2024-11-16 01:49:07+00:00,2024-12-16 08:04:37+00:00,30 days 06:15:30,500,True
3,music_af,214,2025-09-17 13:25:40+00:00,2026-03-25 07:32:42+00:00,188 days 18:07:02,214,False
4,s_john_s,500,2019-03-07 13:17:39+00:00,2019-03-20 12:58:48+00:00,12 days 23:41:09,500,True


## Reconcile the in-memory result with the cleaned CSV

This check reads the generated cleaned file as text and verifies that it matches the approved in-memory logic. It validates raw-field preservation, derived representations, shape, row counts, and keys; it does not write either dataset.


In [10]:
CLEANED_COLUMNS = RAW_COLUMNS + ["listened_at_utc", "artist_mbids_parsed"]
disk_cleaned = pd.read_csv(
    CLEANED_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
)

assert list(disk_cleaned.columns) == CLEANED_COLUMNS
assert disk_cleaned[RAW_COLUMNS].equals(cleaned[RAW_COLUMNS])
assert len(disk_cleaned) == len(cleaned)
assert disk_cleaned["artist_mbids_parsed"].equals(cleaned["artist_mbids_parsed"])
disk_cleaned_utc = pd.to_datetime(
    disk_cleaned["listened_at_utc"], utc=True, errors="coerce"
)
assert disk_cleaned_utc.equals(cleaned["listened_at_utc"])

cleaned_key_duplicates = int(
    cleaned.duplicated(subset=EVENT_KEY_COLUMNS, keep=False).sum()
)
cleaned_missing = pd.Series({
    column: int(
        cleaned[column].isna().sum()
        if column == "listened_at_utc"
        else cleaned[column].eq("").sum()
    )
    for column in cleaned.columns
})

print("Source shape:", raw.shape)
print("Cleaned shape:", cleaned.shape)
print("Raw columns preserved exactly:", disk_cleaned[RAW_COLUMNS].equals(cleaned[RAW_COLUMNS]))
print("Cleaned file matches in-memory raw event rows:", len(disk_cleaned) == len(cleaned))
print("Candidate event key unique:", cleaned_key_duplicates == 0)
print("Timestamp conversion failures:", timestamp_failures)
print("Artist MBID validity:", mbid_validation)
all_missing_columns = list(source_missing.index) + [
    column for column in cleaned_missing.index if column not in source_missing.index
]
missing_comparison = pd.DataFrame({
    "source_missing": source_missing.reindex(all_missing_columns),
    "cleaned_missing": cleaned_missing.reindex(all_missing_columns),
})
print("Missing counts before and after:")
display(missing_comparison)


Source shape: (2214, 9)
Cleaned shape: (2214, 11)
Raw columns preserved exactly: True
Cleaned file matches in-memory raw event rows: True
Candidate event key unique: True
Timestamp conversion failures: 0
Artist MBID validity: {'valid_arrays': 1240, 'invalid_json': 0, 'not_arrays': 0, 'invalid_ids': 0}
Missing counts before and after:


,source_missing,cleaned_missing
username,0.0,0
listened_at,0.0,0
track_name,0.0,0
artist_name,0.0,0
release_name,1027.0,1027
recording_msid,0.0,0
artist_mbids,974.0,974
recording_mbid,943.0,943
release_mbid,974.0,974
listened_at_utc,NaN,0


## Sampling-window limitations

The users' observed windows are different and do not overlap. Four source histories contain exactly 500 rows, matching the sampling cap; one has 214. The window summary is therefore necessary context for any repeat-listening analysis. The dataset is approved for cleaning validation, descriptive exploration, and a dashboard prototype, but it cannot support reliable population-level or synchronized cross-user artist-growth claims. The notebook does not force common windows, infer missing IDs, normalize artist names, or assign final artist/engagement labels.


In [11]:
# Fixed checks for the inspected project sample and generated cleaned output.
assert len(raw) == EXPECTED_ROWS
assert len(cleaned) == EXPECTED_ROWS - exact_duplicate_rows
assert exact_duplicate_rows == 0
assert len(raw.columns) == 9
assert len(cleaned.columns) == 11
assert source_event_key_duplicates == 0
assert cleaned_key_duplicates == 0
assert timestamp_failures == 0
assert mbid_validation["invalid_json"] == 0
assert mbid_validation["not_arrays"] == 0
assert mbid_validation["invalid_ids"] == 0
assert user_observation_summary["username"].nunique() == 5
assert user_observation_summary["row_count"].sum() == len(cleaned)
assert int(user_observation_summary["sampling_cap_reached"].sum()) == 4

print("Validation assertions passed.")


Validation assertions passed.


## Validation summary

Source rows: 2,214  
Cleaned rows: 2,214  
Rows removed: 0  
Exact duplicate rows: 0  
Timestamp conversion failures: 0  
Five record-level checks passed (see `record_checks.md`).
